# causalCalibration Python Workflow

This notebook shows:

- simulate a paper-style heterogeneous treatment effect problem,
- build raw and cross-fitted prediction objects,
- fit ordinary calibration,
- fit cross-calibration in sample,
- compare raw, calibrated, and cross-calibrated predictions,
- interpret diagnostics.


## Setup

The first code cell adds `python/src` to the path so the notebook runs from the repository.


In [ ]:
from pathlib import Path
import importlib.util
import math
import random
import statistics
import sys

repo_root = Path.cwd()
if not (repo_root / "python" / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "python" / "src"))

from causal_calibration import (
    assess_overlap,
    diagnose_calibration,
    fit_calibrator,
    fit_cross_calibrator,
    validate_crossfit_bundle,
)

workflow_method = "isotonic" if (importlib.util.find_spec("lightgbm") is not None and importlib.util.find_spec("sklearn") is not None) else "histogram"

random.seed(11)


## Simulate a paper-style problem

We create a one-dimensional setting with treatment-effect heterogeneity, moderate confounding, and visible miscalibration.


In [ ]:
n = 180
w = [random.uniform(-1.0, 1.0) for _ in range(n)]
propensity = [1.0 / (1.0 + math.exp(-1.6 * value)) for value in w]
treatment = [1.0 if random.random() < p else 0.0 for p in propensity]

tau_true = [0.4 + 0.9 * value + 0.25 * math.sin(3.0 * value) for value in w]
mu0 = [-0.2 + 0.6 * value for value in w]
mu1 = [baseline + effect for baseline, effect in zip(mu0, tau_true)]
outcome_mean = [(1.0 - p) * m0 + p * m1 for p, m0, m1 in zip(propensity, mu0, mu1)]

outcome = []
for a, m0, m1 in zip(treatment, mu0, mu1):
    mean = m1 if a == 1.0 else m0
    noise = random.gauss(0.0, 0.18)
    outcome.append(mean + noise)

tau_raw = [0.55 + 1.35 * effect + 0.25 * (value > 0) - 0.15 * math.cos(4.0 * value) for effect, value in zip(tau_true, w)]

k_folds = 5
fold_ids = [(index % k_folds) + 1 for index in range(n)]
tau_fold_matrix = []
for index, value in enumerate(w):
    row = []
    for fold in range(1, k_folds + 1):
        drift = 0.08 * (fold - 3) + 0.05 * math.sin((fold + 1) * value)
        row.append(tau_raw[index] + drift)
    tau_fold_matrix.append(row)

tau_oof = [row[fold_id - 1] for row, fold_id in zip(tau_fold_matrix, fold_ids)]
tau_new = [tau_raw[index] + 0.03 * math.sin(6.0 * w[index]) for index in range(n)]


## Ordinary calibration

This fits one calibration map from a single prediction per observation.


In [ ]:
overlap = assess_overlap(treatment=treatment, propensity=propensity)
print(overlap.summary())

calibrator = fit_calibrator(
    predictions=tau_raw,
    treatment=treatment,
    outcome=outcome,
    mu0=mu0,
    mu1=mu1,
    propensity=propensity,
    loss="dr",
    method=workflow_method,
)

tau_calibrated = calibrator.predict(tau_new)
print(calibrator.summary())
print(tau_calibrated[:5])


## Cross-calibration

- `predictions=tau_oof` are the pooled out-of-fold predictions used to fit the calibration map.
- `fold_predictions=tau_fold_matrix` is the full `n x K` matrix of fold-specific predictions used for calibrated aggregation.


In [ ]:
validate_crossfit_bundle(
    predictions=tau_oof,
    fold_predictions=tau_fold_matrix,
    fold_ids=fold_ids,
)

cross_calibrator = fit_cross_calibrator(
    predictions=tau_oof,
    fold_predictions=tau_fold_matrix,
    fold_ids=fold_ids,
    treatment=treatment,
    outcome=outcome,
    mu0=mu0,
    mu1=mu1,
    propensity=propensity,
    loss="dr",
    method=workflow_method,
)

tau_cross_calibrated = cross_calibrator.predict(tau_fold_matrix)
print(cross_calibrator.summary())
print(tau_cross_calibrated[:5])


## Compare raw, calibrated, and cross-calibrated predictions

In [ ]:
comparison = [
    {
        "raw": round(tau_raw[index], 3),
        "ordinary": round(calibrator.predict([tau_raw[index]])[0], 3),
        "cross": round(tau_cross_calibrated[index], 3),
        "true": round(tau_true[index], 3),
    }
    for index in range(5)
]
comparison


## Diagnostics

Here we compare the raw predictor to the cross-calibrated one and read the BLP slope diagnostic.


In [ ]:
diagnostics = diagnose_calibration(
    predictions=tau_cross_calibrated,
    comparison_predictions=tau_raw,
    treatment=treatment,
    outcome=outcome,
    mu0=mu0,
    mu1=mu1,
    propensity=propensity,
    curve_method="histogram",
    fold_ids=fold_ids,
    target_population="both",
)

diagnostics.summary()
diagnostics.blp_diagnostic.summary()


## What population is being calibrated?

The example above uses `loss="dr"`, which targets calibration in the original study population.

If overlap is weak, `loss="r"` may be more robust because it emphasizes the overlap region through the R-loss weighting. The tradeoff is that the target becomes overlap-weighted rather than the original study population.

- choose `dr` when original-population targeting matters and overlap is acceptable,
- choose `r` when overlap is weak and overlap-weighted targeting is acceptable.
